# 01 — Cleaning and Splitting

**Purpose.** Turn raw data into clean train and test splits, using only
deterministic, model-agnostic rules.

**Inputs.** `data/raw/` and the findings from notebook 00.

**Outputs.** `data/processed/train.parquet`, `data/processed/test.parquet`,
both DVC-tracked.

---

### The boundary rule — the most important rule in this project

**Allowed here** (deterministic, model-agnostic):
- Schema and dtype validation
- Duplicate removal
- Dropping non-predictive columns (record IDs and the like)
- Dropping records with a missing or invalid label
- Dropping records that violate a **hard, externally known constraint**
- The train/test split

**Forbidden here** (learns from the data — belongs in `02x`):
- Imputation of missing values
- Statistical outlier detection (IQR, z-score, isolation forest)
- Scaling, encoding, any fitted transformation
- Feature engineering and selection

| Type | Criterion | Where |
|---|---|---|
| Invalid record | Violates a bound known *before* seeing the data | **01** |
| Statistical outlier | Threshold derived *from* the data | **02x** |

Any data-derived threshold applied here is computed over train *and* test, and
leaks test-set information into the filtering decision.

**This notebook is a thin wrapper over `src/data/clean.py`.** The same sequence
runs unattended as `task clean:data` and as the `clean_split` DVC stage. Put
logic in the module, narration here.

## 1. Setup

In [ ]:
# Standard setup for every notebook in this project.
# Autoreload so edits in src/ take effect without restarting the kernel.
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from src.config import load_config
from src.utils.seed import set_seed

cfg = load_config()
set_seed(cfg.seed)

pd.set_option("display.max_columns", 50)
cfg

## 2. Load the raw data

In [ ]:
from src.data.load import load_raw

df_raw = load_raw(cfg)
audit = [("raw", len(df_raw))]
print(f"{len(df_raw):,} records loaded")
df_raw.head()

## 3. Validate against the declared schema

Fail here rather than three notebooks later. `configs/data.yaml` declares the
contract; `src/data/schema.py` enforces it.

In [ ]:
from src.data import schema

df = schema.validate(df_raw, cfg, strict=False)
schema.find_violations(df, cfg).head(20)

## 4. Remove duplicates

Decide explicitly what "duplicate" means for this dataset — fully identical
rows, or rows identical on a natural key — and record the choice.

In [ ]:
from src.data.clean import drop_duplicates

df = drop_duplicates(df, cfg)
audit.append(("after dedup", len(df)))
print(f"{audit[-2][1] - len(df):,} duplicate records removed")

## 5. Drop records with a missing or invalid label

Unlabelled records cannot be trained or scored on, so dropping them is
model-agnostic. Report the share: a large fraction is a data-collection problem
worth surfacing, not something to quietly discard.

In [ ]:
from src.data.clean import drop_invalid_labels

df = drop_invalid_labels(df, cfg)
audit.append(("after label filter", len(df)))
print(f"{audit[-2][1] - len(df):,} records dropped for missing/invalid labels")

## 6. Apply hard constraints

⚠️ **Only bounds with a documented external source.** Each rule applied here is
declared in `configs/data.yaml` with its `source` field: the standard,
specification or physical limit that defines it.

If you are about to write `df[df.x < df.x.quantile(0.99)]`, stop — that is a
statistical threshold. It belongs in a `02x` notebook, fitted on train only.

In [ ]:
from src.data.clean import drop_constraint_violations

df = drop_constraint_violations(df, cfg)
audit.append(("after hard constraints", len(df)))
print(f"{audit[-2][1] - len(df):,} records violated a hard constraint")

## 7. Drop non-predictive columns

Identifiers let a model memorise records and often encode collection order.
Drop them after deduplication, which may need them — and keep the grouping
column until after the split.

In [ ]:
from src.data.clean import drop_non_predictive

df = drop_non_predictive(df, cfg)
df.columns.tolist()

## 8. Cleaning audit

Row counts per step. This table is the evidence that cleaning did what section
10 of notebook 00 specified — paste it into the report.

In [ ]:
audit_df = pd.DataFrame(audit, columns=["step", "records"])
audit_df["removed"] = -audit_df["records"].diff().fillna(0).astype(int)
audit_df["share_remaining"] = audit_df["records"] / audit[0][1]
audit_df

## 9. Re-validate

The same contract, checked against the cleaned frame. Passing here proves the
cleaning code actually enforced what the config declares.

In [ ]:
df = schema.validate(df, cfg, strict=True)
print("schema OK")

## 10. Train/test split

From `src/data/split.py` — the single authoritative splitter — using the method,
grouping column and seed in `configs/data.yaml`. Never call a splitter from
`sklearn` directly here: models validated on different folds cannot be compared.

In [ ]:
from src.data.split import train_test_split

train_df, test_df = train_test_split(df, cfg)
print(f"train: {len(train_df):,}   test: {len(test_df):,}"
      f"   ({len(test_df) / len(df):.1%} test)")

## 11. Leakage check

Cheap, and it catches the most expensive class of bug in the project: results
that look good and are wrong.

In [ ]:
from src.data.split import assert_no_group_leakage

assert_no_group_leakage(train_df, test_df, cfg)
print("no group appears in both splits")

# Sanity check the split is representative — a large distribution gap between
# train and test usually means the grouping column carries structure.
pd.DataFrame({
    "train": train_df[cfg.target].describe(),
    "test": test_df[cfg.target].describe(),
})

## 12. Persist and version

Write both splits, then track them with DVC so the exact contents are pinned to
this Git commit:

```bash
dvc add data/processed/train.parquet data/processed/test.parquet
git add data/processed/*.dvc configs/data.yaml
git commit -m "Clean and split dataset"
```

Or let the pipeline do it: `task dvc:repro` runs the same steps through
`src/data/clean.py`.

In [ ]:
from src.data.load import save_processed

save_processed(train_df, cfg.train_path)
save_processed(test_df, cfg.test_path)
print(f"wrote {cfg.train_path} and {cfg.test_path}")

## 13. Handoff checklist

Before moving to a `02x` notebook:

- [ ] Every filter applied here is deterministic and model-agnostic
- [ ] Every hard bound has a documented `source` in `configs/data.yaml`
- [ ] No imputation, scaling, encoding or statistical outlier removal happened
- [ ] The leakage assertion passed
- [ ] Both splits are written and `dvc add`-ed
- [ ] The cleaning audit is recorded

**The test split is now frozen. Nothing reads it again until notebook 03.**